# Advanced Hybrid RAG for Financial-QA-10k

This notebook builds a complete RAG pipeline for the Kaggle **Financial-QA-10k** dataset.

It includes:

- Dataset cleaning
- Adaptive chunking based on your `context_words` distribution
- BM25 keyword retrieval
- Gemini embeddings
- Qdrant Cloud vector search
- Hybrid retrieval using Reciprocal Rank Fusion
- Gemini-based reranking
- Gemini answer generation with citations
- Retrieval evaluation using Hit@K
- MLflow experiment tracking

Expected original dataset path:

```python
dataset/Financial-QA-10k.csv
```

You can change `DATASET_PATH` in the configuration cell if your file is located somewhere else.

## 1. Install packages

In [ ]:
!pip install -U pandas numpy scikit-learn tqdm python-dotenv
!pip install -U google-genai qdrant-client rank-bm25 mlflow

## 2. Imports

In [ ]:
import os
import re
import json
import time
import getpass
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from rank_bm25 import BM25Okapi

from google import genai
from google.genai import types

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import mlflow

## 3. Configuration

In [ ]:
# =========================
# API KEYS
# =========================

GEMINI_API_KEY = getpass.getpass("Enter Gemini API Key: ")
QDRANT_API_KEY = getpass.getpass("Enter Qdrant API Key: ")
QDRANT_URL = input("Enter Qdrant Cloud URL: ").strip()

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["QDRANT_API_KEY"] = QDRANT_API_KEY
os.environ["QDRANT_URL"] = QDRANT_URL


# =========================
# DATASET PATH
# Change this if your file path is different.
# =========================

DATASET_PATH = r"dataset/Financial-QA-10k.csv"


# =========================
# MODEL CONFIG
# =========================

# Official Gemini embedding model name.
EMBED_MODEL = "gemini-embedding-001"

# 768 is a good cost/performance setting for RAG.
EMBED_DIM = 768

# Generation model. Change if needed based on your Gemini account/model availability.
GEN_MODEL = "gemini-2.5-flash"


# =========================
# FOLDER CONFIG
# =========================

PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(exist_ok=True)


# =========================
# DATASET CONFIG
# =========================

REQUIRED_COLS = ["question", "answer", "context", "ticker", "filing"]


# =========================
# CHUNKING CONFIG
# Your context_words stats:
# count: 6740
# mean: 33.06
# median: 27
# 75%: 40
# max: 787
#
# Therefore: keep most contexts as-is, split only long outliers.
# =========================

CHUNK_MAX_WORDS = 150
CHUNK_OVERLAP_WORDS = 25


# =========================
# INDEX CONFIG
# Start with 500 for testing.
# After successful run, set INDEX_LIMIT = None for full indexing.
# =========================

INDEX_LIMIT = 500

COLLECTION_NAME = "financial_qa_10k"

RECREATE_QDRANT_COLLECTION = True


# =========================
# RETRIEVAL CONFIG
# =========================

BM25_TOP_K = 30
VECTOR_TOP_K = 30
HYBRID_TOP_K = 10
FINAL_TOP_K = 5

RRF_K = 60

USE_GEMINI_RERANKER = True

## 4. Create Gemini and Qdrant clients

In [ ]:
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

qdrant_client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)

print("Gemini client created.")
print("Qdrant client created.")

try:
    print(qdrant_client.get_collections())
except Exception as e:
    print("Qdrant connection error:", e)
    raise

## 5. Load dataset

In [ ]:
dataset_path = Path(DATASET_PATH)

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATASET_PATH}\\n"
        "Please update DATASET_PATH in the configuration cell."
    )

df = pd.read_csv(dataset_path)

print("Loaded:", dataset_path)
print("Shape:", df.shape)

df.head()

## 6. Validate columns

In [ ]:
missing_cols = [col for col in REQUIRED_COLS if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("All required columns exist.")
print(df.columns.tolist())

## 7. Clean text fields

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text


for col in REQUIRED_COLS:
    df[col] = df[col].apply(clean_text)

df.head()

## 8. Remove bad rows

In [ ]:
before = len(df)

df = df.dropna(subset=["question", "answer", "context"])
df = df[df["question"].str.len() > 5]
df = df[df["answer"].str.len() > 1]
df = df[df["context"].str.len() > 30]

df = df.reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Removed:", before - after)

## 9. Add IDs and metadata

These fields are important because Qdrant stores vectors as points. Each point needs a stable ID and payload metadata for citations.

- `row_id`: original row reference
- `context_hash`: used for duplicate detection and evaluation
- `doc_id`: readable document ID

In [ ]:
def make_hash(text):
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()


df["row_id"] = df.index.astype(str)
df["context_hash"] = df["context"].apply(make_hash)

df["doc_id"] = (
    df["ticker"].str.lower()
    + "_"
    + df["filing"].str.lower()
    + "_"
    + df["context_hash"].str[:10]
)

df[["row_id", "doc_id", "ticker", "filing", "question", "answer", "context"]].head()

## 10. Check context word distribution

In [ ]:
def word_count(text):
    return len(str(text).split())


df["context_words"] = df["context"].apply(word_count)

df["context_words"].describe()

## 11. Adaptive chunking

In [ ]:
def word_window_chunks(text, max_words=150, overlap_words=25):
    words = str(text).split()

    if len(words) <= max_words:
        return [str(text)]

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap_words

    return chunks


def make_adaptive_chunks(text, max_words=150, overlap_words=25):
    text = clean_text(text)

    if word_count(text) <= max_words:
        return [text]

    return word_window_chunks(
        text=text,
        max_words=max_words,
        overlap_words=overlap_words
    )

## 12. Create chunked documents

In [ ]:
parent_docs_df = df.drop_duplicates(subset=["context_hash"]).copy()
parent_docs_df = parent_docs_df.reset_index(drop=True)

chunk_records = []
point_id = 0

for _, row in parent_docs_df.iterrows():
    chunks = make_adaptive_chunks(
        row["context"],
        max_words=CHUNK_MAX_WORDS,
        overlap_words=CHUNK_OVERLAP_WORDS
    )

    for chunk_index, chunk_text in enumerate(chunks):
        chunk_hash = make_hash(chunk_text)

        chunk_records.append({
            "point_id": point_id,
            "chunk_id": f"{row['doc_id']}_chunk_{chunk_index}",
            "parent_doc_id": row["doc_id"],
            "row_id": row["row_id"],
            "ticker": row["ticker"],
            "filing": row["filing"],
            "source": "Financial-QA-10k",

            "chunk_text": chunk_text,
            "chunk_hash": chunk_hash,
            "chunk_index": chunk_index,
            "chunk_words": word_count(chunk_text),

            "source_context": row["context"],
            "source_context_hash": row["context_hash"],

            "original_question": row["question"],
            "original_answer": row["answer"]
        })

        point_id += 1

docs_df = pd.DataFrame(chunk_records)

print("Original unique contexts:", len(parent_docs_df))
print("Total chunks created:", len(docs_df))
print("Extra chunks due to long contexts:", len(docs_df) - len(parent_docs_df))

docs_df.head()

## 13. Check chunk distribution

In [ ]:
print(docs_df["chunk_words"].describe())

print("Chunks over 150 words:", (docs_df["chunk_words"] > 150).sum())
print("Single-chunk parent docs:", docs_df.groupby("parent_doc_id").size().eq(1).sum())
print("Multi-chunk parent docs:", docs_df.groupby("parent_doc_id").size().gt(1).sum())

## 14. Save processed files

In [ ]:
df.to_csv(PROCESSED_DIR / "financial_qa_clean.csv", index=False)
parent_docs_df.to_csv(PROCESSED_DIR / "financial_qa_parent_documents.csv", index=False)
docs_df.to_csv(PROCESSED_DIR / "financial_qa_chunked_documents.csv", index=False)

print("Processed files saved in:", PROCESSED_DIR.resolve())

## 15. Train / validation / test split

In [ ]:
train_eval_df, test_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42
)

train_df, val_df = train_test_split(
    train_eval_df,
    test_size=0.10,
    random_state=42
)

train_df.to_csv(PROCESSED_DIR / "train_questions.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val_questions.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_questions.csv", index=False)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

## 16. Select sample or full dataset for indexing

In [ ]:
if INDEX_LIMIT is None:
    index_docs_df = docs_df.copy()
else:
    index_docs_df = docs_df.head(INDEX_LIMIT).copy()

index_docs_df = index_docs_df.reset_index(drop=True)
index_docs_df["point_id"] = index_docs_df.index.astype(int)

print("Chunks selected for indexing:", len(index_docs_df))
index_docs_df.head()

## 17. Build BM25 keyword index

In [ ]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9_.$%-]+", " ", text)
    return text.split()


bm25_corpus = index_docs_df["chunk_text"].tolist()
tokenized_corpus = [tokenize(doc) for doc in bm25_corpus]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created.")

## 18. BM25 search function

In [ ]:
def bm25_search(query, top_k=5):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        row = index_docs_df.iloc[idx]

        results.append({
            "point_id": int(row["point_id"]),
            "rank": rank,
            "score": float(scores[idx]),
            "ticker": row["ticker"],
            "filing": row["filing"],
            "row_id": row["row_id"],
            "chunk_id": row["chunk_id"],
            "context": row["chunk_text"],
            "source_context_hash": row["source_context_hash"],
            "source": row["source"],
            "retrieval_method": "bm25"
        })

    return results


# Test BM25
test_query = "What area did NVIDIA initially focus on before expanding into other markets?"

bm25_results = bm25_search(test_query, top_k=3)

for r in bm25_results:
    print("Score:", r["score"])
    print("Ticker:", r["ticker"], "| Filing:", r["filing"], "| Row:", r["row_id"])
    print(r["context"][:500])
    print("-" * 80)

## 19. Gemini embedding functions

In [ ]:
def format_document_for_embedding(row):
    return f'''
Task: Represent this 10-K financial filing context for retrieval.
Ticker: {row['ticker']}
Filing: {row['filing']}
Text: {row['chunk_text']}
'''.strip()


def format_query_for_embedding(query):
    return f'''
Task: Represent this financial question for retrieving relevant 10-K filing context.
Question: {query}
'''.strip()


def embed_text(text, retries=3, sleep_seconds=2):
    for attempt in range(retries):
        try:
            result = gemini_client.models.embed_content(
                model=EMBED_MODEL,
                contents=text,
                config=types.EmbedContentConfig(
                    output_dimensionality=EMBED_DIM
                )
            )

            return result.embeddings[0].values

        except Exception as e:
            print(f"Embedding error attempt {attempt + 1}: {e}")
            time.sleep(sleep_seconds)

    raise RuntimeError("Embedding failed after retries.")


# Test one embedding
sample_row = index_docs_df.iloc[0]
sample_embedding = embed_text(format_document_for_embedding(sample_row))

print("Embedding dimension:", len(sample_embedding))
print(sample_embedding[:5])

## 20. Generate or load document embeddings

In [ ]:
embedding_cache_path = PROCESSED_DIR / f"embeddings_{EMBED_MODEL}_{EMBED_DIM}_{len(index_docs_df)}.npy"

if embedding_cache_path.exists():
    embeddings = np.load(embedding_cache_path)
    print("Loaded cached embeddings:", embeddings.shape)

else:
    embeddings = []

    for _, row in tqdm(index_docs_df.iterrows(), total=len(index_docs_df)):
        text = format_document_for_embedding(row)
        emb = embed_text(text)
        embeddings.append(emb)

        # Basic rate-limit protection
        time.sleep(0.2)

    embeddings = np.array(embeddings, dtype=np.float32)
    np.save(embedding_cache_path, embeddings)

    print("Generated and saved embeddings:", embeddings.shape)

## 21. Create Qdrant collection

In [ ]:
if RECREATE_QDRANT_COLLECTION:
    if qdrant_client.collection_exists(COLLECTION_NAME):
        qdrant_client.delete_collection(COLLECTION_NAME)
        print("Old collection deleted.")

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=EMBED_DIM,
            distance=Distance.COSINE
        )
    )
    print("Collection created:", COLLECTION_NAME)
else:
    print("Collection already exists:", COLLECTION_NAME)

## 22. Upload vectors to Qdrant Cloud

In [ ]:
def upload_to_qdrant(index_docs_df, embeddings, batch_size=100):
    total = len(index_docs_df)

    for start in tqdm(range(0, total, batch_size)):
        end = min(start + batch_size, total)
        batch_points = []

        for i in range(start, end):
            row = index_docs_df.iloc[i]

            payload = {
                "text": row["chunk_text"],
                "ticker": row["ticker"],
                "filing": row["filing"],
                "row_id": str(row["row_id"]),
                "chunk_id": row["chunk_id"],
                "chunk_index": int(row["chunk_index"]),
                "parent_doc_id": row["parent_doc_id"],
                "source_context_hash": row["source_context_hash"],
                "source": row["source"]
            }

            point = PointStruct(
                id=int(row["point_id"]),
                vector=embeddings[i].tolist(),
                payload=payload
            )

            batch_points.append(point)

        qdrant_client.upsert(
            collection_name=COLLECTION_NAME,
            points=batch_points,
            wait=True
        )

    print("Uploaded points:", total)


upload_to_qdrant(index_docs_df, embeddings, batch_size=100)

## 23. Qdrant vector search

In [ ]:
def vector_search(query, top_k=5):
    query_text = format_query_for_embedding(query)
    query_embedding = embed_text(query_text)

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=top_k,
        with_payload=True
    )

    results = []

    for rank, point in enumerate(response.points, start=1):
        payload = point.payload

        results.append({
            "point_id": int(point.id),
            "rank": rank,
            "score": float(point.score),
            "ticker": payload.get("ticker"),
            "filing": payload.get("filing"),
            "row_id": payload.get("row_id"),
            "chunk_id": payload.get("chunk_id"),
            "context": payload.get("text"),
            "source_context_hash": payload.get("source_context_hash"),
            "source": payload.get("source"),
            "retrieval_method": "vector"
        })

    return results


# Test vector search
vector_results = vector_search(test_query, top_k=3)

for r in vector_results:
    print("Score:", r["score"])
    print("Ticker:", r["ticker"], "| Filing:", r["filing"], "| Row:", r["row_id"])
    print(r["context"][:500])
    print("-" * 80)

## 24. Hybrid search using Reciprocal Rank Fusion

In [ ]:
def reciprocal_rank_fusion(bm25_results, vector_results, rrf_k=60):
    fused = {}

    for result in bm25_results:
        pid = result["point_id"]
        rank = result["rank"]

        if pid not in fused:
            fused[pid] = result.copy()
            fused[pid]["rrf_score"] = 0.0
            fused[pid]["bm25_rank"] = None
            fused[pid]["vector_rank"] = None
            fused[pid]["bm25_score"] = 0.0
            fused[pid]["vector_score"] = 0.0

        fused[pid]["rrf_score"] += 1 / (rrf_k + rank)
        fused[pid]["bm25_rank"] = rank
        fused[pid]["bm25_score"] = result["score"]

    for result in vector_results:
        pid = result["point_id"]
        rank = result["rank"]

        if pid not in fused:
            fused[pid] = result.copy()
            fused[pid]["rrf_score"] = 0.0
            fused[pid]["bm25_rank"] = None
            fused[pid]["vector_rank"] = None
            fused[pid]["bm25_score"] = 0.0
            fused[pid]["vector_score"] = 0.0

        fused[pid]["rrf_score"] += 1 / (rrf_k + rank)
        fused[pid]["vector_rank"] = rank
        fused[pid]["vector_score"] = result["score"]

    final_results = list(fused.values())
    final_results = sorted(final_results, key=lambda x: x["rrf_score"], reverse=True)

    for i, result in enumerate(final_results, start=1):
        result["rank"] = i
        result["retrieval_method"] = "hybrid_rrf"

    return final_results

## 25. Gemini reranker

In [ ]:
def extract_json_array(text):
    text = str(text).strip()

    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)
    text = text.strip()

    match = re.search(r"\[.*\]", text, flags=re.DOTALL)

    if match:
        text = match.group(0)

    return json.loads(text)


def gemini_rerank(question, candidates, top_n=5):
    if not candidates:
        return []

    candidate_blocks = []

    for c in candidates:
        block = f'''
Candidate ID: {c['point_id']}
Ticker: {c['ticker']}
Filing: {c['filing']}
Context: {c['context']}
'''
        candidate_blocks.append(block)

    prompt = f'''
You are a financial retrieval reranker.

Question:
{question}

Candidates:
{chr(10).join(candidate_blocks)}

Task:
Rank the candidates by how well they answer the question.

Return ONLY a JSON array of Candidate IDs in best-to-worst order.
Example:
[12, 5, 9]
'''

    try:
        response = gemini_client.models.generate_content(
            model=GEN_MODEL,
            contents=prompt
        )

        ranked_ids = extract_json_array(response.text)
        ranked_ids = [int(x) for x in ranked_ids]

        candidate_map = {int(c["point_id"]): c for c in candidates}

        reranked = []

        for pid in ranked_ids:
            if pid in candidate_map:
                reranked.append(candidate_map[pid])

        existing_ids = set([r["point_id"] for r in reranked])

        for c in candidates:
            if c["point_id"] not in existing_ids:
                reranked.append(c)

        for i, r in enumerate(reranked, start=1):
            r["rerank_position"] = i
            r["retrieval_method"] = "hybrid_rrf_gemini_rerank"

        return reranked[:top_n]

    except Exception as e:
        print("Gemini reranker failed. Falling back to hybrid order.")
        print("Error:", e)
        return candidates[:top_n]

## 26. Final hybrid retrieval function

In [ ]:
def hybrid_search(
    query,
    final_top_k=5,
    bm25_top_k=30,
    vector_top_k=30,
    hybrid_top_k=10,
    use_reranker=True
):
    bm25_results = bm25_search(query, top_k=bm25_top_k)
    vector_results = vector_search(query, top_k=vector_top_k)

    hybrid_candidates = reciprocal_rank_fusion(
        bm25_results=bm25_results,
        vector_results=vector_results,
        rrf_k=RRF_K
    )

    hybrid_candidates = hybrid_candidates[:hybrid_top_k]

    if use_reranker:
        final_results = gemini_rerank(
            question=query,
            candidates=hybrid_candidates,
            top_n=final_top_k
        )
    else:
        final_results = hybrid_candidates[:final_top_k]

    return final_results


# Test hybrid retrieval
hybrid_results = hybrid_search(
    test_query,
    final_top_k=FINAL_TOP_K,
    bm25_top_k=BM25_TOP_K,
    vector_top_k=VECTOR_TOP_K,
    hybrid_top_k=HYBRID_TOP_K,
    use_reranker=USE_GEMINI_RERANKER
)

for r in hybrid_results:
    print("Rank:", r.get("rerank_position", r.get("rank")))
    print("RRF Score:", r.get("rrf_score"))
    print("BM25 Rank:", r.get("bm25_rank"), "| Vector Rank:", r.get("vector_rank"))
    print("Ticker:", r["ticker"], "| Filing:", r["filing"], "| Row:", r["row_id"])
    print(r["context"][:500])
    print("-" * 80)

## 27. Build citations

In [ ]:
def build_context_block(results):
    blocks = []

    for i, r in enumerate(results, start=1):
        block = f'''
[Source {i}]
Ticker: {r['ticker']}
Filing: {r['filing']}
Row ID: {r['row_id']}
Chunk ID: {r['chunk_id']}
Source: {r['source']}
Context: {r['context']}
'''
        blocks.append(block)

    return "\n".join(blocks)


def build_citation_table(results):
    citation_rows = []

    for i, r in enumerate(results, start=1):
        citation_rows.append({
            "source_number": f"Source {i}",
            "ticker": r["ticker"],
            "filing": r["filing"],
            "row_id": r["row_id"],
            "chunk_id": r["chunk_id"],
            "source": r["source"],
            "context_preview": r["context"][:250]
        })

    return pd.DataFrame(citation_rows)

## 28. Generate answer with citations

In [ ]:
def answer_with_citations(question, retrieved_results):
    context_block = build_context_block(retrieved_results)

    prompt = f'''
You are a financial RAG assistant.

Answer the user's question using ONLY the provided sources.

Question:
{question}

Sources:
{context_block}

Rules:
- Give a concise answer.
- Include citations like [Source 1], [Source 2].
- Do not use outside knowledge.
- If the answer is not present in the sources, say:
  "I don't know based on the provided context."
'''

    response = gemini_client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt
    )

    return response.text


# Test full RAG
question = "What area did NVIDIA initially focus on before expanding into other markets?"

retrieved = hybrid_search(
    question,
    final_top_k=FINAL_TOP_K,
    bm25_top_k=BM25_TOP_K,
    vector_top_k=VECTOR_TOP_K,
    hybrid_top_k=HYBRID_TOP_K,
    use_reranker=USE_GEMINI_RERANKER
)

answer = answer_with_citations(question, retrieved)

print("ANSWER:")
print(answer)

print("\nCITATIONS:")
citation_df = build_citation_table(retrieved)
citation_df

## 29. Reusable RAG function

In [ ]:
def ask_financial_rag(question):
    retrieved = hybrid_search(
        question,
        final_top_k=FINAL_TOP_K,
        bm25_top_k=BM25_TOP_K,
        vector_top_k=VECTOR_TOP_K,
        hybrid_top_k=HYBRID_TOP_K,
        use_reranker=USE_GEMINI_RERANKER
    )

    answer = answer_with_citations(question, retrieved)
    citations = build_citation_table(retrieved)

    return {
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
        "citations": citations
    }


result = ask_financial_rag("What does the filing say about the company's main business focus?")

print(result["answer"])
result["citations"]

## 30. Retrieval evaluation: Hit@K

In [ ]:
def retrieval_hit_at_k(question_row, search_fn, top_k=5):
    question = question_row["question"]
    correct_context_hash = question_row["context_hash"]

    results = search_fn(question, top_k=top_k)

    retrieved_parent_hashes = [
        r["source_context_hash"]
        for r in results
    ]

    return int(correct_context_hash in retrieved_parent_hashes)


def search_for_eval(query, top_k=5):
    return hybrid_search(
        query,
        final_top_k=top_k,
        bm25_top_k=BM25_TOP_K,
        vector_top_k=VECTOR_TOP_K,
        hybrid_top_k=HYBRID_TOP_K,
        use_reranker=USE_GEMINI_RERANKER
    )

## 31. Prepare evaluation sample

In [ ]:
indexed_context_hashes = set(index_docs_df["source_context_hash"].tolist())

eval_sample = val_df[
    val_df["context_hash"].isin(indexed_context_hashes)
].head(50).copy()

print("Evaluation sample size:", len(eval_sample))
eval_sample.head()

## 32. Run retrieval evaluation

In [ ]:
hits = []

for _, row in tqdm(eval_sample.iterrows(), total=len(eval_sample)):
    hit = retrieval_hit_at_k(
        question_row=row,
        search_fn=search_for_eval,
        top_k=FINAL_TOP_K
    )

    hits.append(hit)

hit_at_k = float(np.mean(hits)) if hits else 0.0

print(f"Hit@{FINAL_TOP_K}:", hit_at_k)

## 33. Generate answers for small evaluation sample

In [ ]:
answer_eval_rows = []

small_answer_eval = eval_sample.head(10).copy()

for _, row in tqdm(small_answer_eval.iterrows(), total=len(small_answer_eval)):
    question = row["question"]
    ground_truth = row["answer"]

    rag_output = ask_financial_rag(question)

    answer_eval_rows.append({
        "question": question,
        "ground_truth_answer": ground_truth,
        "rag_answer": rag_output["answer"],
        "ticker": row["ticker"],
        "filing": row["filing"]
    })

answer_eval_df = pd.DataFrame(answer_eval_rows)

answer_eval_df.to_csv(PROCESSED_DIR / "answer_eval_sample.csv", index=False)

answer_eval_df.head()

## 34. Log experiment in MLflow

In [ ]:
mlflow.set_experiment("financial_qa_10k_advanced_rag")

with mlflow.start_run(run_name="gemini_qdrant_hybrid_rerank"):
    mlflow.log_param("dataset", "Financial-QA-10k")
    mlflow.log_param("num_original_rows", len(df))
    mlflow.log_param("num_unique_contexts", len(parent_docs_df))
    mlflow.log_param("num_chunks_total", len(docs_df))
    mlflow.log_param("num_chunks_indexed", len(index_docs_df))

    mlflow.log_param("chunking_method", "adaptive_context_as_chunk")
    mlflow.log_param("chunk_max_words", CHUNK_MAX_WORDS)
    mlflow.log_param("chunk_overlap_words", CHUNK_OVERLAP_WORDS)

    mlflow.log_param("embedding_model", EMBED_MODEL)
    mlflow.log_param("embedding_dim", EMBED_DIM)
    mlflow.log_param("generation_model", GEN_MODEL)

    mlflow.log_param("vector_db", "Qdrant Cloud")
    mlflow.log_param("collection_name", COLLECTION_NAME)

    mlflow.log_param("bm25_top_k", BM25_TOP_K)
    mlflow.log_param("vector_top_k", VECTOR_TOP_K)
    mlflow.log_param("hybrid_top_k", HYBRID_TOP_K)
    mlflow.log_param("final_top_k", FINAL_TOP_K)
    mlflow.log_param("hybrid_method", "reciprocal_rank_fusion")
    mlflow.log_param("rrf_k", RRF_K)
    mlflow.log_param("reranker", "gemini" if USE_GEMINI_RERANKER else "none")

    mlflow.log_metric(f"hit_at_{FINAL_TOP_K}", hit_at_k)

    mlflow.log_artifact(str(PROCESSED_DIR / "financial_qa_clean.csv"))
    mlflow.log_artifact(str(PROCESSED_DIR / "financial_qa_chunked_documents.csv"))

    if (PROCESSED_DIR / "answer_eval_sample.csv").exists():
        mlflow.log_artifact(str(PROCESSED_DIR / "answer_eval_sample.csv"))

print("MLflow run logged.")

## 35. Start MLflow UI

In [ ]:
# Run this cell, then open http://localhost:5000
!mlflow ui --port 5000

## 36. Final manual testing

In [ ]:
questions = [
    "What area did NVIDIA initially focus on before expanding into other markets?",
    "What does the company say about revenue?",
    "What are the company's main risks?",
    "What does the filing say about liquidity?",
    "How does the company describe its business strategy?"
]

for q in questions:
    print("=" * 100)
    print("QUESTION:", q)

    output = ask_financial_rag(q)

    print("\nANSWER:")
    print(output["answer"])

    print("\nCITATIONS:")
    display(output["citations"])

## Notes

For first run:

```python
INDEX_LIMIT = 500
```

After everything works, change it to:

```python
INDEX_LIMIT = None
```

Then rerun from **Cell 16 onward**.

Your dataset has very short contexts, so this notebook uses:

```text
Adaptive chunking
Keep context as one chunk if <= 150 words
Split only long contexts using 150-word chunks with 25-word overlap
```